# Evaluate Saved PPO Models and Compare Benchmarks

This notebook loads the three PPO models produced by the final pipeline, evaluates them again on the untouched 2023–2024 test period, runs MVO/DJI/Equal Weight, and displays the six-strategy financial comparison.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from stable_baselines3 import PPO

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(path for path in (cwd, cwd.parent) if (path / 'src').is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RESULTS_DIR, TRAINING_DEVICE
from src.environment import PortfolioEnv
from src.evaluator import Evaluator
from src.main import PORTFOLIOS, _metrics_table, evaluate_benchmarks, prepare_experiment_data
from src.utils import plot_cumulative_returns


In [ ]:
RUNS_ROOT = RESULTS_DIR / 'final_evaluation'
available_runs = sorted((path for path in RUNS_ROOT.glob('run_*') if path.is_dir()), key=lambda path: path.stat().st_mtime)
if not available_runs:
    raise FileNotFoundError('No completed final run exists. Run `python -m src.main` first.')

# Change this assignment if you want to inspect an older run.
RUN_DIRECTORY = available_runs[-1]
manifest = json.loads((RUN_DIRECTORY / 'manifest.json').read_text(encoding='utf-8'))

print('Evaluating run:', RUN_DIRECTORY.name)
print('Training device:', manifest['device'])
print('Benchmark portfolio:', manifest['benchmark_portfolio'])

In [ ]:
# Rebuilds states and applies a scaler fitted on training data only.
data = prepare_experiment_data()
print('Out-of-sample dates:', data['test_raw']['Date'].min(), 'to', data['test_raw']['Date'].max())

In [ ]:
evaluations = {}
models_directory = RUN_DIRECTORY / 'models'

for portfolio in PORTFOLIOS:
    model_path = models_directory / f'ppo_{portfolio.lower()}'
    test_environment = PortfolioEnv(
        data['test_scaled'],
        portfolio,
        price_data=data['test_raw'],
    )
    model = PPO.load(model_path, env=test_environment, device=TRAINING_DEVICE)
    evaluations[f'PPO {portfolio}'] = Evaluator(model, test_environment).evaluate()
    print(f'Loaded and evaluated PPO {portfolio}')

In [ ]:
evaluations.update(evaluate_benchmarks(data, manifest['benchmark_portfolio']))
assert list(evaluations) == [
    'PPO Conservative', 'PPO Moderate', 'PPO Aggressive',
    'MVO', 'DJI', 'Equal Weight',
]
print('All six strategies evaluated.')

In [ ]:
financial_metrics = _metrics_table(evaluations)
display(financial_metrics.style.format('{:.2%}', subset=['Annual Return', 'Cumulative Return', 'Annual Volatility', 'Maximum Drawdown']).format('{:.3f}', subset=['Sharpe Ratio']))

In [ ]:
figure, axis = plot_cumulative_returns(evaluations)
axis.set_title('Out-of-Sample Cumulative Returns: PPO vs Benchmarks')
figure.show()